# 男女播音员 · 中英混读 tiny-voca 训练
从 MyDrive/tiny-voca-mix/downloads 读取独立标注包与男女声音频包，在运行环境本地解压后训练。默认男女各 31,799 段，共 63,598 段，沿用原句的训练、验证和测试划分。Misaki 提供音素和重音，MMS_FA 为每段录音单独估计音素时长。训练检查点可切换男女声。


In [ ]:
#@title 训练参数
SEED = 20261113 #@param {type:"integer"}
PREPROCESS_WORKERS = 4 #@param {type:"integer"}
ALIGN_BATCH_SIZE = 4 #@param {type:"integer"}
SMOKE = False #@param {type:"boolean"}
BATCH_FRAMES = 4000 #@param {type:"integer"}
MAX_BATCH = 16 #@param {type:"integer"}

if type(SEED) is not int or not 0 <= SEED < 2**63:
    raise ValueError("SEED 必须是 0 到 2**63-1 之间的整数")
if type(SMOKE) is not bool:
    raise ValueError("SMOKE 必须为布尔值")
for name, value in (("PREPROCESS_WORKERS", PREPROCESS_WORKERS), ("ALIGN_BATCH_SIZE", ALIGN_BATCH_SIZE), ("BATCH_FRAMES", BATCH_FRAMES), ("MAX_BATCH", MAX_BATCH)):
    if type(value) is not int or value < 1:
        raise ValueError(f"{name} 必须是正整数")
STEPS = 80 if SMOKE else 104000


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, tempfile, zipfile

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/tiny-voca-mix')
DOWNLOADS = ROOT / 'downloads'
WORK = Path('/content/tiny-voca-announcers')
DATA = WORK / 'paired'
OUTPUT = ROOT / 'tiny-voca-training'
WORK.mkdir(exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

from contextlib import contextmanager
from datetime import datetime, timezone
import shlex
import time
import threading
import traceback
import uuid

log_dir = OUTPUT / 'logs'
log_dir.mkdir(exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
LOG = log_dir / f'run-{stamp}-{uuid.uuid4().hex[:8]}.log'

class DriveLog:
    def __init__(self, path):
        self.path = path
        self.pending = []
        self.lock = threading.RLock()
        self.timer = None

    def flush(self):
        with self.lock:
            if self.timer:
                self.timer.cancel()
                self.timer = None
            if self.pending:
                with self.path.open('a', encoding='utf-8') as stream:
                    stream.writelines(self.pending)
                self.pending.clear()

    def write(self, message, *, immediate=True):
        line = f'[{datetime.now(timezone.utc).isoformat(timespec="seconds")}] {message}'
        print(line, flush=True)
        with self.lock:
            self.pending.append(line + '\n')
            if immediate or len(self.pending) >= 128:
                self.flush()
            elif self.timer is None:
                self.timer = threading.Timer(5, self.flush)
                self.timer.daemon = True
                self.timer.start()

log = DriveLog(LOG).write

@contextmanager
def stage(name):
    started = time.monotonic()
    log(f'开始 {name}')
    try:
        yield
    except BaseException:
        log(f'{name} 耗时 {time.monotonic()-started:.2f}s，中断\n{traceback.format_exc()}')
        raise
    else:
        log(f'完成 {name}，耗时 {time.monotonic()-started:.2f}s')

def run(*args):
    command = list(map(str, args))
    started = time.monotonic()
    log('$ ' + shlex.join(command))
    env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONFAULTHANDLER='1')
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          env=env, text=True, errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                log(line.rstrip(), immediate=False)
            status = process.wait()
        except BaseException:
            if process.poll() is None:
                process.kill()
            process.wait()
            raise
    log(f'耗时 {time.monotonic()-started:.2f}s，退出码 {status}' + (f' (signal {-status})' if status < 0 else ''))
    if status:
        raise subprocess.CalledProcessError(status, command)

log(f'本次运行日志 {LOG}')

def unpack_dataset(downloads, work):
    labels = downloads / 'tiny-voca-mix-colab-31799.zip'
    audio = {s: downloads / f'tiny-voca-{s}-31799.zip' for s in ('male', 'female')}
    stamps = []
    for path in (labels, *audio.values()):
        stat = path.stat()
        stamps.append((path.name, stat.st_size, stat.st_mtime_ns))
    version = hashlib.sha256(json.dumps(stamps).encode()).hexdigest()[:16]
    dest = work / f'dataset-{version}'
    if (dest / 'announcers.json').is_file():
        log(f'复用本地解压目录 {dest}')
        return dest

    with tempfile.TemporaryDirectory(dir=work) as tmp:
        stage = Path(tmp) / 'dataset'
        inputs = stage / 'inputs'
        inputs.mkdir(parents=True)
        with zipfile.ZipFile(labels) as z:
            for name in ('corpus.jsonl', 'misaki.jsonl'):
                with z.open('tiny-voca-mix-qwen3-tts/inputs/' + name) as src, (inputs / name).open('wb') as out:
                    shutil.copyfileobj(src, out)
        runs = {}
        for speaker, path in audio.items():
            local = Path(tmp) / path.name
            log(f'复制到 Colab 本地磁盘 {path.name}')
            shutil.copyfile(path, local)
            with zipfile.ZipFile(local) as z:
                prefix = speaker + '/'
                recipe = z.read(prefix + 'recipe.json')
                rid = hashlib.sha256(recipe).hexdigest()[:16]
                folder = stage / 'runs' / rid
                folder.mkdir(parents=True)
                (folder / 'recipe.json').write_bytes(recipe)
                manifest = z.read(prefix + 'corpus.jsonl')
                (folder / 'corpus.jsonl').write_bytes(manifest)
                records = [json.loads(line) for line in manifest.splitlines()]
                names = {r['audio'] for r in records}
                if len(names) != 31799 or len(records) != len(names):
                    raise ValueError(f'{speaker} 录音清单数量异常')
                members = [i for i in z.infolist() if i.filename.endswith('.wav')]
                if len(members) != len(names) or {i.filename for i in members} != {prefix + n for n in names}:
                    raise ValueError(f'{speaker} ZIP 与录音清单不匹配')
                for i, member in enumerate(members, 1):
                    rel = member.filename[len(prefix):]
                    if not re.fullmatch(r'audio/(zh|en|mixed)/[0-9a-f]{2}/[A-Za-z0-9_-]+\.wav', rel):
                        raise ValueError(rel)
                    target = folder / rel
                    target.parent.mkdir(exist_ok=True, parents=True)
                    with z.open(member) as src, target.open('wb') as out:
                        shutil.copyfileobj(src, out)
                    if i % 5000 == 0 or i == len(members):
                        log(f'{speaker} 本地解压 {i:,}/{len(members):,}')
                runs[speaker] = rid
            local.unlink()
        (stage / 'announcers.json').write_text(json.dumps(runs))
        stage.rename(dest)
    return dest

run('nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv')


## 安装独立训练环境
使用 Python 3.11、Torch/TorchAudio 2.8、Misaki 中文 v1.1 和美式英语前端。

In [ ]:
def choose_wheel(capability, driver_cuda):
    if capability >= (10, 0):
        if driver_cuda >= (12, 8):
            return 'cu128'
        raise RuntimeError(f'当前 GPU 架构 {capability} 需要支持 CUDA 12.8 的驱动')
    if driver_cuda >= (12, 6):
        return 'cu126'
    if driver_cuda >= (11, 8):
        return 'cu118'
    raise RuntimeError(f'当前驱动报告 CUDA {driver_cuda}，无法使用此训练环境')

gpu_info = subprocess.check_output([
    'nvidia-smi', '-i', '0', '--query-gpu=name,compute_cap,driver_version',
    '--format=csv,noheader'
], text=True).strip()
name, capability, driver = map(str.strip, gpu_info.split(','))
banner = subprocess.check_output(['nvidia-smi'], text=True)
match = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', banner)
if not match:
    raise RuntimeError('无法从 nvidia-smi 获取驱动支持的 CUDA 版本')
wheel = choose_wheel(tuple(map(int, capability.split('.'))), tuple(map(int, match.groups())))
log(f'GPU {name} | 计算能力 {capability} | 驱动 {driver} | {match[0]} | PyTorch {wheel}')
run(sys.executable,'-m','pip','install','-q','uv')
venv = WORK / f'env-{wheel}'
PY = venv / 'bin/python'
if not PY.exists():
    run('uv','venv','--python','3.11',venv)
run('uv','pip','install','--python',PY,'--index-url',f'https://download.pytorch.org/whl/{wheel}',
    f'torch==2.8.0+{wheel}', f'torchaudio==2.8.0+{wheel}')
requirements = WORK/'requirements-misaki.txt'
requirements.write_text('# Shared Chinese and American English frontend for training and inference.\nmisaki[zh]==0.9.4\npypinyin==0.55.0\npypinyin-dict==0.9.0\njieba==0.42.1\ncn2an==0.5.24\nnum2words==0.5.14\nspacy==3.7.5\nnumpy==1.26.4\ntyper==0.12.5\nclick==8.1.8\nsetuptools<81\nphonemizer-fork==3.3.2\nespeakng-loader==0.2.4\nen-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl\nregex==2026.9.29\n', encoding='utf-8')
run('uv','pip','install','--python',PY,'-r',requirements,'librosa==0.11.0','soundfile==0.13.1')
run(PY, '-c', '''
import torch, torchaudio
from torchaudio.functional import forced_align
x = torch.ones((32, 32), device='cuda', requires_grad=True)
(x @ x).mean().backward()
assert x.grad.isfinite().all().item()
scores = torch.tensor([[[0., -4.], [-4., 0.], [0., -4.]]], device='cuda')
alignment, _ = forced_align(scores, torch.tensor([[1]], dtype=torch.int32, device='cuda'))
assert alignment.tolist() == [[0, 1, 0]]
print('CUDA 反向传播与音频强制对齐验证通过', torch.__version__, torchaudio.__version__, torch.cuda.get_device_name())
''')
with stage('解压训练数据'):
    LOCAL_ROOT = unpack_dataset(DOWNLOADS, WORK)


## 训练代码与双语前端
训练标注和新检查点的推理调用同一份 Misaki 适配代码。

In [ ]:
import base64, io
SRC = WORK/'source'
SRC.mkdir(exist_ok=True)
bundle = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(bundle))) as z:
    z.extractall(SRC)
os.environ['PYTHONPATH']=str(SRC)

In [ ]:
run(PY, '-c', 'from tiny_voca.text import frontend_identity, frontend_fingerprint; print(frontend_identity()["phonemizer"], frontend_fingerprint())')

In [ ]:
# 预处理脚本已随训练源码解压到 SRC/preprocess.py。


## 配对男女声录音与全量音素标注
按原句 ID 核对两套生成结果，每个音色使用独立的录音 ID。

In [ ]:
with stage('配对录音'):
    run(PY, '-c', 'from pathlib import Path; import sys; from prepare_announcers import prepare; from tiny_voca.text import frontend_fingerprint; prepare(Path(sys.argv[1]), Path(sys.argv[2]), frontend_fingerprint())', LOCAL_ROOT, DATA)


## 提取特征与强制对齐
每块完成即存到 Drive。重连时会恢复到本地磁盘。按语种输出接受量；被拒样本的 ID 和原因保存在 preparation 下。

In [ ]:
with stage('预处理与强制对齐'):
    if SMOKE:
        subset=WORK/'announcers-smoke'
        subset.mkdir(exist_ok=True)
        shutil.copy2(DATA/'prepare-config.json',subset/'prepare-config.json')
        for split in ('train','validation','test'):
            rows=list(map(json.loads,(DATA/(split+'.jsonl')).read_text().splitlines()))
            count=8 if split=='train' else 2
            picked=[r for voice in ('male','female') for lang in ('zh','en','mixed')
                    for r in [v for v in rows if v['language']==lang and v['voice_id']==voice][:count]]
            (subset/(split+'.jsonl')).write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in picked))
        INPUT_DATA=subset
    else:
        INPUT_DATA=DATA
    FEATURES=WORK/('features-smoke' if SMOKE else 'features')
    run(PY,SRC/'preprocess.py',INPUT_DATA,FEATURES,OUTPUT,PREPROCESS_WORKERS,ALIGN_BATCH_SIZE)

## 训练与续训
默认每 500 步保存到 Drive。改变 batch 或其他训练配方时应使用新实验名。中途重连后重跑前面的单元，再运行本单元即可续训。

In [ ]:
def experiment_for_seed(output, annotation, seed):
    base = output / ('model-' + annotation[:12])
    seeded = output / f'{base.name}-seed-{seed}'
    checkpoints = base / 'checkpoints'
    recipe = checkpoints / 'recipe.json'
    if not seeded.exists() and (checkpoints / 'last.pt').is_file():
        previous_seed = json.loads(recipe.read_text())['seed'] if recipe.is_file() else 20261113
        if previous_seed == seed:
            return base
    return seeded

with stage('训练'):
    import shutil
    annotation=json.loads((FEATURES/'annotations.json').read_text())['fingerprint']
    experiment = experiment_for_seed(OUTPUT, annotation, SEED)
    experiment.mkdir(exist_ok=True)
    snapshot_id = hashlib.sha256(bundle.encode()).hexdigest()
    marker = experiment/'source'/'bundle.sha256'
    if not marker.exists() or marker.read_text() != snapshot_id:
        shutil.copytree(SRC, experiment/'source', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
        marker.write_text(snapshot_id)
    RUN=experiment/'checkpoints'
    last=RUN/'last.pt'
    args=[PY,'-m','tiny_voca.train','--config',DATA/'prepare-config.json','--device','cuda',
          '--train',FEATURES/'train.jsonl','--validation',FEATURES/'validation.jsonl',
          '--output',RUN,'--steps',STEPS,'--batch-frames',BATCH_FRAMES,'--max-batch',MAX_BATCH,
          '--seed',SEED,'--log-every',10,'--save-every',500,'--keep-every',10000,'--max-seconds',86400,'--no-preload']
    if last.exists():
        args.extend(['--resume',last])
    log(f'SEED {SEED} | 检查点 {last} | 恢复 {last.exists()} | 目标 {STEPS}')
    if not last.exists() and RUN.exists() and all(f.name in ('recipe.json','config.json','metrics.jsonl') for f in RUN.iterdir()):
        RUN.rename(experiment/('interrupted-before-first-save-'+datetime.now().strftime('%Y%m%d%H%M%S%f')))
    run(*args, '--skip-complete')


## 用同一个检查点分别试听男女声


In [ ]:
from IPython.display import Audio, display
checkpoint = RUN / 'inference.pt'
for voice in ('male', 'female'):
    for lang,text in [('zh','今天最低气温是负十度。'),('en','Could you open the window, please?'),('mixed','这个 weekend 我想去 hiking，你来吗？')]:
        wav=WORK/(voice+'-'+lang+'.wav')
        run(PY,'-m','tiny_voca.infer','--checkpoint',checkpoint,'--voice',voice,'--text',text,'--output',wav)
        print(voice, text)
        display(Audio(filename=str(wav)))